In [ ]:
import matplotlib.pyplot as plt
import albumentations as A
from albumentations.pytorch import ToTensorV2
import numpy as np
from PIL import Image
import torch
from transformers import Sam3Processor, Sam3Model, pipeline
from huggingface_hub import login
import cv2

from google.colab import userdata

Setting up the transforms pipeline.

In [ ]:
def get_train_transforms_v2():
    """
    Returns transformations for the training set.
    Includes augmentations suitable for cell/cytology images (rotation invariant)
    and resizing to 1008x1008 for optimal SAM3 backbone alignment.
    """
    return A.Compose([
        # Resize to 1008x1008 to match SAM3 patch size (14 * 72 = 1008)
        # This avoids padding issues in the model.
        A.Resize(height=1008, width=1008),

        # Cytology images are usually rotation invariant
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),

        A.Affine(
          scale=(0.95, 1.05),      # Scale between 95% and 105%
          translate_percent=0.02,  # Shift up to 5%
          rotate=(-5, 5),        # Rotation between -15 and 15 degrees
          shear=(-1.2, 1.2),           # Very slight stretching (mimics slide tilt)
          p=0.2
        ),

        A.GaussNoise(std_range=(0.075, 0.12), p=0.4),

        A.CoarseDropout(
          num_holes_range=(1, 8),
          hole_height_range=(10, 28),
          hole_width_range=(10, 28),
          fill=0,
          p=0.1
        ),

        # RB and RC: Adjusts lighting and sharpness of your PAP smear images
        A.RandomBrightnessContrast(
            brightness_limit=0.12,
            contrast_limit=0.12,
            p=0.2
        ),

        # Normalize to 0-1 and convert to Tensor
        # Note: Mean/Std normalization is typically handled by the FasterRCNN model internal transform
        # checking against ImageNet stats. We output 0-1 float tensors here.
        A.ToFloat(max_value=255.0),
        ToTensorV2()
    ]
                     #, bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels'])
    )

Testing full SAM3 segment everything capabilities on training set image. Including masks detected after transformations and before them ratio calculation.  

In [ ]:
# Authenticate with Hugging Face (Replace with your token)
login(token=userdata.get('hugging_face_token'))

# Initialize the Automatic Mask Generation (AMG) pipeline
# device=0 uses the first GPU. Change to device=-1 to use CPU.
device = 0 if torch.cuda.is_available() else -1
print("Loading SAM 3 Segment Everything pipeline...")

generator = pipeline("mask-generation", model="facebook/sam3", device=device)

# Load your local image
image_path = "./LSIL_6_2.png" # Change to your image path
image = Image.open(image_path).convert("RGB")

image_np = np.array(image)

# Get your pipeline 
# Note: Ensure your transform does NOT include Normalize() for this specific pipeline
transforms = get_train_transforms_v2()
for i in range(32):
  transformed = transforms(image=image_np)
  transformed_image_np = transformed["image"]

  # If your transform outputs a Tensor, convert it back to a uint8 numpy array
  if isinstance(transformed_image_np, torch.Tensor):
      # Move to CPU, change from (C, H, W) to (H, W, C), and scale to 0-255
      transformed_image_np = (transformed_image_np.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
      transformed_image_pil = Image.fromarray(transformed_image_np).convert("RGB")

  # RUN INFERENCE
  # We pass the transformed image directly
  outputs_transformed = generator(transformed_image_pil, points_per_batch=64)
  outputs = generator(image, points_per_batch=64)

  masks_transformed = outputs_transformed["masks"]
  masks = outputs["masks"]
  print(f"Generated {len(masks)} distinct masks for non-transformed vs {len(masks_transformed)} for transformed.")
  print(f"Ratio is {len(masks_transformed) / len(masks)}")

  # Visualization (Multi-color overlay)
  # Convert PIL image to OpenCV format (BGR)
  image_cv = cv2.cvtColor(transformed_image_np, cv2.COLOR_RGB2BGR)

  # Create a blank canvas to accumulate the colored masks
  mask_canvas = np.zeros_like(image_cv)

  # Loop through each generated mask
  for mask in masks_transformed:
      # Generate a random BGR color for this specific mask
      random_color = np.random.randint(0, 255, size=(3,), dtype=int).tolist()

      # The pipeline outputs boolean masks. Convert to numpy.
      mask_np = np.array(mask)

      # Apply the random color to the mask area on our canvas
      mask_canvas[mask_np > 0] = random_color

  # Blend the original image with the colorful mask canvas
  # alpha=0.6 means original image is 60% visible, masks are 40% visible
  final_image = cv2.addWeighted(image_cv, 0.6, mask_canvas, 0.4, 0)

  # Save and display the result
  cv2.imwrite(f"sam3_transformed_everything_{i}.jpg", final_image)
  print(f"Saved final visualization to 'sam3_transformed_everything_{i}.jpg'")

In [ ]:
# Load the image
image_path = "./LSIL_6_2.png"
image = cv2.imread(image_path)

if image is None:
    raise FileNotFoundError(f"Could not find the image at {image_path}. Please check the path.")

# Define the raw data provided from the train.csv
raw_data = """LSIL_6_2.png,906.7480916030536,995.3384223918576,100,100,LSIL,4
LSIL_6_2.png,31.267175572519083,911.9592875318068,100,100,HSIL,5
LSIL_6_2.png,312.67175572519085,729.5674300254453,100,100,HSIL,5
LSIL_6_2.png,317.882951653944,846.8193384223919,100,100,HSIL,5
LSIL_6_2.png,190.2086513994912,776.4681933842239,100,100,NILM,0
LSIL_6_2.png,333.5165394402036,898.9312977099237,100,100,HSIL,5
LSIL_6_2.png,302.24936386768445,935.409669211196,100,100,HSIL,5
LSIL_6_2.png,257.9541984732825,914.5648854961833,100,100,HSIL,5
LSIL_6_2.png,78.16793893129771,562.8091603053437,100,100,HSIL,5
LSIL_6_2.png,211.05343511450383,469.00763358778624,100,100,HSIL,5
LSIL_6_2.png,857.2417302798982,502.88040712468194,100,100,HSIL,5
LSIL_6_2.png,479.43002544529264,403.86768447837153,100,100,LSIL,4"""

# Define a color dictionary (BGR format for OpenCV)
class_colors = {
    "NILM": (0, 255, 0),    # Green
    "LSIL": (0, 165, 255),  # Orange
    "HSIL": (0, 0, 255)     # Red
}

# Process each line of the data
lines = raw_data.strip().split('\n')

for line in lines:
    parts = line.split(',')

    # Extract coordinates and dimensions (converting floats to integers for pixel indexing)
    x = int(float(parts[1]))
    y = int(float(parts[2]))
    w = int(float(parts[3]))
    h = int(float(parts[4]))

    # Extract class info
    class_name = parts[5]

    # Get the color for this class (default to white if not found)
    color = class_colors.get(class_name, (255, 255, 255))

    # Calculate bottom-right corner
    x2 = x + w
    y2 = y + h

    # Draw the bounding box
    cv2.rectangle(image, (x, y), (x2, y2), color, thickness=2)

    # Add the class label above the box
    # Create a filled rectangle as background for the text to make it readable
    text = class_name
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.6
    font_thickness = 2

    # Get text size
    (text_w, text_h), baseline = cv2.getTextSize(text, font, font_scale, font_thickness)

    # Draw text background
    cv2.rectangle(image, (x, y - text_h - 5), (x + text_w, y), color, -1)

    # Put text (in white or black depending on color contrast)
    text_color = (255, 255, 255) if class_name == "HSIL" else (0, 0, 0)
    cv2.putText(image, text, (x, y - 5), font, font_scale, text_color, font_thickness)

# Save and display the result
output_filename = "LSIL_6_2_visualized.png"
cv2.imwrite(output_filename, image)
print(f"Image saved as {output_filename}")